### WIYN Open Cluster Study. XCVII. An Extended Radial-velocity Survey and Spectroscopic Binary Orbits in the Open Cluster NGC 188

Ritvik Sai Narayan, Evan Linck, Robert D. Mathieu, and Aaron M. Geller

https://iopscience.iop.org/article/10.3847/1538-3881/ae2d14


They provide 35 new spectroscopic-binary orbits from their (RV) survey of the old (6.4 ± 0.2 Gyr) open cluster NGC 188. 

They also find several BSS systems 

"We find that NGC 188 has 18 secure BSSs:
WOCS 2679, 4230, 4290, 4306, 4348, 4535, 4581, 4589, 4970,
5078, 5325, 5350, 5379, 5434, 5467, 5885, 5934, and 8104.
Four other stars in the BSS region of the color–magnitude
diagram (CMD), WOCS 4447, 4540, 4945, and 5020, "

This notebook is to crossmatch those systems with the new orbital solutions in their Table 6, and then update systems in our catalog if needed. 

In [20]:
import numpy as np
import json
import pandas as pd

import astropy.units as u
from astroquery.vizier import Vizier
import re

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR

In [21]:
BSS_IDS = [2679, 4230, 4290, 4306, 4348, 4535, 4581, 4589, 4970,
5078, 5325, 5350, 5379, 5434, 5467, 5885, 5934, 8104]
BSS_Candidates = [4447, 4540, 4945, 5020]


In [22]:
col_names = [
    "PKM", "Per", "e_Per", "Ncyc", "gamma", "e_gamma", "K", "e_K",
    "e", "e_e", "omega", "e_omega", "T0", "e_T0", "asini",
    "e_asini", "fm", "e_fm", "sig", "Nobs"
 ]

table_6 = pd.read_csv(DATA_DIR / "from_others" / "R_Narayan26.txt",
    sep=r"\s+",engine="python", skiprows=30,names=col_names,)

# WOCS IDs are integers;
table_6["PKM"] = table_6["PKM"].astype(int)

In [23]:
display(table_6.head())
print(f"len {len(table_6)}")

,PKM,Per,e_Per,Ncyc,gamma,e_gamma,K,e_K,e,e_e,omega,e_omega,T0,e_T0,asini,e_asini,fm,e_fm,sig,Nobs
0,269,553.00000,3.00000,11.3,-42.10,0.40,7.10,0.90,0.720,0.050,69.0,12.0,55313.00,17.00,38.00,6.00,0.0070,0.0030,0.886,15
1,583,119.78000,0.04000,55.1,-41.22,0.14,6.27,0.22,0.180,0.040,64.0,12.0,55474.00,4.00,10.20,0.40,0.0029,0.0003,0.618,20
2,3719,51.57450,0.00160,128.6,-43.06,0.22,28.10,0.40,0.562,0.008,225.8,1.4,56022.17,0.12,16.46,0.24,0.0670,0.0030,0.959,23
3,3755,10.32601,0.00009,642.2,-42.60,0.30,39.10,0.40,0.206,0.011,72.0,3.0,55602.72,0.08,5.43,0.06,0.0598,0.0018,1.083,19
4,3953,940.40000,2.50000,7.1,-41.14,0.14,4.51,0.19,0.040,0.040,240.0,70.0,56340.00,170.00,58.20,2.50,0.0089,0.0011,0.633,24


len 35


In [24]:
targets_secure = set(BSS_IDS)
targets_candidates = set(BSS_Candidates)
targets_all = targets_secure | targets_candidates

matches = table_6[table_6["PKM"].isin(targets_all)].copy().sort_values("PKM")
matches["is_secure_BSS"] = matches["PKM"].isin(targets_secure)
matches["is_candidate"] = matches["PKM"].isin(targets_candidates)

display(matches[["PKM", "Per", "e", "fm", "Nobs", "is_secure_BSS", "is_candidate"]])
print(f"Matched IDs: {matches['PKM'].tolist()}")


# missing_secure = sorted(targets_secure - set(matches["PKM"]))
# missing_candidates = sorted(targets_candidates - set(matches["PKM"]))
# print(f"Missing secure BSS IDs: {missing_secure}")
# print(f"Missing candidate IDs: {missing_candidates}")

,PKM,Per,e,fm,Nobs,is_secure_BSS,is_candidate
10,4230,0.455609,0.04,0.0039,32,True,False
18,4945,5110.000000,0.48,0.1400,15,False,True
19,5020,38000.000000,0.74,0.0400,27,False,True
32,8104,3460.000000,0.12,0.0520,25,True,False


Matched IDs: [4230, 4945, 5020, 8104]


### Next we wan't to create a json table for these systems, to be added to the master table.

WOCS 4230: A Blue Straggler with a close WD companion, which is a potential W UMa system, with a period of 0.456 days.
WOCS 4945: The faintest Blue Straggler in NGC 188, which is a long period system with P ~5000 days.
WOCS 5020: A Blue Straggler with the longest known period for a cluster BSS, with a period of ~38,000 days. 


In [25]:
# ----------------------------------------------------------------------------
# Build catalog entries for the selected BSS systems
# Coordinates + pos_err_mas from SIMBAD; orbital parameters from Table 6.
# Catalog coordinate format: scalar RA/Dec in deg, pos_err_mas in milliarcsec.
# ----------------------------------------------------------------------------
sys.path.insert(0, str(proj_root / "code" / "data_compilation"))
from Get_Coords_From_SIMBAD import get_coords

# The 3 focus systems (WOCS 8104 is a secure BSS too; flip the toggle to include it)
INCLUDE_WOCS_8104 = False
FOCUS_IDS = [4230, 4945, 5020] + ([8104] if INCLUDE_WOCS_8104 else [])

NOTES = {
    4230: "BSS with a close WD companion, which is a potential W UMa system, with a period of 0.456 days.",
    4945: "The faintest BSS in NGC 188, which is a long period system with P about 5000 days.",
    5020: "A Blue Straggler with the longest known period for a cluster BSS, with a period of 38,000 days.",
    8104: "Secure BSS in NGC 188 with a long-period orbit (P about 3460 days).",
}

# SIMBAD lookup (NGC 188 WOCS/Platais IDs resolve as 'Cl* NGC 188 PKM NNNN')
simbad = {wocs: coords for wocs, coords in zip(
    FOCUS_IDS, get_coords([f"Cl* NGC 188 PKM {wocs}" for wocs in FOCUS_IDS]))}

def triplet(row, val_col, err_col):
    """[err-, value, err+] from a Table 6 row; None where missing."""
    val = pd.to_numeric(row[val_col], errors="coerce")
    err = pd.to_numeric(row[err_col], errors="coerce")
    val = None if pd.isna(val) else float(val)
    err = None if pd.isna(err) else float(err)
    return [err, val, err]

json_dict = []
for wocs in FOCUS_IDS:
    row = matches[matches["PKM"] == wocs].iloc[0]
    coords = simbad[wocs]
    json_dict.append({
        "System Name": f"WOCS {wocs}",
        "RA": round(coords["ra_deg"], 7),          # deg
        "Dec": round(coords["dec_deg"], 7),        # deg
        "pos_err_mas": round(float(coords["pos_err_mas"]), 4),  # 1-sigma on-sky error, mas
        "Reference": ["2026AJ....171..102N"],
        "Notes": NOTES[wocs] + f" Alternative name: {' '.join(coords['name'].split())} (SIMBAD).",
        "Period": triplet(row, "Per", "e_Per"),            # days
        "Eccentricity": triplet(row, "e", "e_e"),
        "M1": [None, None, None],                          # no component masses in Table 6 (SB1)
        "M2": [None, None, None],
        "Mass Function": triplet(row, "fm", "e_fm"),       # Msun
        "M1_sin3i": [None, None, None],
        "M2_sin3i": [None, None, None],
        "system_class": "Blue straggler binary",
        "obs_type_1": None,
        "obs_type_2": None,
        "evol_type_1": "MS",
        "evol_type_2": "WD",
        "Detection Method": ["RV", "SB1"],
        "quality_flags": [],
    })

for entry in json_dict:
    assert entry["RA"] is not None and 10 < entry["RA"] < 13, entry      # NGC 188: RA ~ 11.8 deg
    assert 85 < entry["Dec"] < 85.5, entry                               # NGC 188: Dec ~ +85.24
    assert entry["Period"][1] is not None, entry

display(pd.DataFrame([{k: e[k] for k in ("System Name", "RA", "Dec", "pos_err_mas", "Period", "Eccentricity", "Mass Function")} for e in json_dict]))

,System Name,RA,Dec,pos_err_mas,Period,Eccentricity,Mass Function
0,WOCS 4230,10.849565,85.342413,0.0174,"[1e-06, 0.455609, 1e-06]","[0.03, 0.04, 0.03]","[0.0003, 0.0039, 0.0003]"
1,WOCS 4945,12.077625,85.292575,0.0392,"[50.0, 5110.0, 50.0]","[0.07, 0.48, 0.07]","[0.07, 0.14, 0.07]"
2,WOCS 5020,11.964943,85.252533,0.0127,"[7000.0, 38000.0, 7000.0]","[0.16, 0.74, 0.16]","[0.05, 0.04, 0.05]"


In [ ]:
# Save as a raw ingest table (Combine_and_process_data.py globs raw_json/*.json)
out_path = RAW_JSON_DIR / "BSS_NGC188_Narayan26.raw.json"
with open(out_path, "w") as f:
    json.dump(json_dict, f, indent=2)
print(f"Wrote {len(json_dict)} systems to {out_path}")


Wrote 3 systems to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/BSS_NGC188_Narayan26.raw.json
